In [5]:
!pip install pandas requests librosa soundfile tqdm


Active code page: 1252
Defaulting to user installation because normal site-packages is not writeable


In [6]:
import os
import pandas as pd
import requests
import json
import librosa
import soundfile as sf
from tqdm import tqdm


In [7]:
BASE_PATH = r"C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\data"

CSV_PATH = os.path.join(BASE_PATH, "raw_inputs", "FT_Data_with_fixed_urls.csv")

AUDIO_DIR = os.path.join(BASE_PATH, "downloaded_audio")
TRANSCRIPT_DIR = os.path.join(BASE_PATH, "downloaded_transcripts")
PROCESSED_DIR = os.path.join(BASE_PATH, "processed_training_data")

# Create folders if not exist
os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(TRANSCRIPT_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Folders ready ✅")


Folders ready ✅


In [8]:
df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()

print("Total rows:", len(df))
df.head(3)


Total rows: 104


,user_id,recording_id,language,duration,rec_url_gcp,transcription_url_gcp,metadata_url_gcp,fixed_audio_url,fixed_trans_url
0,245746,825780,hi,443,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/upload_goai/967...,https://storage.googleapis.com/upload_goai/967...
1,291038,825727,hi,443,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/upload_goai/967...,https://storage.googleapis.com/upload_goai/967...
2,246004,988596,hi,475,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/upload_goai/114...,https://storage.googleapis.com/upload_goai/114...


In [9]:
import os
import pandas as pd
import requests
import json
import librosa
import soundfile as sf
from tqdm import tqdm

BASE_PATH = r"C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\data"

DATA_CSV = os.path.join(BASE_PATH, "raw_inputs", "FT_Data_with_fixed_urls.csv")

AUDIO_DIR = os.path.join(BASE_PATH, "downloaded_audio")
TRANSCRIPT_DIR = os.path.join(BASE_PATH, "downloaded_transcripts")
PROCESSED_DIR = os.path.join(BASE_PATH, "processed_training_data")

os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(TRANSCRIPT_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

df = pd.read_csv(DATA_CSV)

training_rows = []

for _, row in tqdm(df.iterrows(), total=len(df)):

    audio_url = row["fixed_audio_url"]
    transcript_url = row["fixed_trans_url"]
    recording_id = row["recording_id"]

    audio_path = os.path.join(AUDIO_DIR, f"{recording_id}.wav")
    transcript_path = os.path.join(TRANSCRIPT_DIR, f"{recording_id}.json")

    # Download audio
    if not os.path.exists(audio_path):
        r = requests.get(audio_url)
        with open(audio_path, "wb") as f:
            f.write(r.content)

    # Load full audio once
    audio, sr = librosa.load(audio_path, sr=16000)

    # Download transcript
    if not os.path.exists(transcript_path):
        r = requests.get(transcript_url)
        with open(transcript_path, "wb") as f:
            f.write(r.content)

    with open(transcript_path, "r", encoding="utf-8") as f:
        segments = json.load(f)

    # 🔥 Segment-wise processing
    for i, seg in enumerate(segments):

        start = seg["start"]
        end = seg["end"]
        text = seg["text"].strip()

        if len(text) == 0:
            continue

        start_sample = int(start * 16000)
        end_sample = int(end * 16000)

        segment_audio = audio[start_sample:end_sample]

        segment_filename = f"{recording_id}_seg_{i}.wav"
        segment_path = os.path.join(PROCESSED_DIR, segment_filename)

        sf.write(segment_path, segment_audio, 16000)

        training_rows.append({
            "audio_path": segment_path,
            "text": text
        })

# Save new training dataset
output_csv = os.path.join(PROCESSED_DIR, "training_data_segmented.csv")
pd.DataFrame(training_rows).to_csv(output_csv, index=False)

print("Segment-wise preprocessing complete ✅")
print("Total training samples:", len(training_rows))


100%|██████████| 104/104 [11:27<00:00,  6.61s/it]


Segment-wise preprocessing complete ✅
Total training samples: 5941


In [11]:
output_csv = os.path.join(PROCESSED_DIR, "training_data_segmented.csv")

pd.DataFrame(training_rows).to_csv(output_csv, index=False)

print("Saved at:", output_csv)


Saved at: C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\data\processed_training_data\training_data_segmented.csv
